## Data helper functions (used by all notebooks)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from pandas import DataFrame, Series
from enum import StrEnum, auto
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler # used in CV

RANDOM_STATE = 42

TrainTestData = tuple[DataFrame, Series, DataFrame, DataFrame, Series, Series]
Model = LogisticRegression | SVC | RandomForestClassifier
Metric = dict[str, float | int]
AllMetrics = dict[str, dict]

constTargetAndMetadata = ['tnbc', 'case_id']
evalTargetN = 'nTNBC'
evalTarget = 'TNBC'
targetNames = [evalTargetN, evalTarget]

class FeatureVariant(StrEnum):
    LITERATURE = auto()
    BORUTA = auto()
    RFE = auto()
    LASSO = auto()
    KOTHARI = auto()
    STATISTICAL = auto()
    AUTOMATED = auto()

    def print_info():
        print([key for key in FeatureVariant.__members__])

class ModelVariant(StrEnum):
    SVM = 'svm'
    RF = 'random_forest'
    LG = 'logistic_regression'

    def print_info():
        print([key for key in ModelVariant.__members__])


def split_data(df: DataFrame, target: str, case_id=None) -> TrainTestData:

    # Features: all columns except target column
    X: DataFrame = df.drop(columns=[target])
    # Target variable
    y: Series = df[target]

    return capstone_train_test_split(X, y, case_id)

def apply_smote_to_train(X_train: DataFrame, y_train: Series):
    sm: SMOTE = SMOTE(random_state=42) # can have different parameters
    X_train, y_train = sm.fit_resample(X_train, y_train)

    return X_train, y_train

def split_data_apply_smote(df: DataFrame, target: str) -> TrainTestData:

    # Features: all columns except target column
    X_input: DataFrame = df.drop(columns=[target, 'case_id'])
    # Target variable
    y_input: Series = df[target]

    # first split data
    X, y, X_train, X_test, y_train, y_test, test_case_ids = capstone_train_test_split(X_input, y_input)

    sm: SMOTE = SMOTE(random_state=42) # can have different parameters
    X_train, y_train = sm.fit_resample(X_train, y_train)

    return X, y, X_train, X_test, y_train, y_test, test_case_ids

def capstone_train_test_split(X: DataFrame, y: Series, contains_case_id: bool = False) -> TrainTestData:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    # Take out case ID but keep then available for testing data (for initial validation)
    if contains_case_id:
        case_id: str = "case_id"
        test_case_id: Series = X_test[case_id]
        X.drop(columns=[case_id], inplace=True)
        X_train.drop(columns=[case_id], inplace=True)
        X_test.drop(columns=[case_id], inplace=True)
    else:
        test_case_id = None

    # Training size = 0.8 * 977 ≈ 781
    # Test size = 0.2 * 977 ≈ 196
    print(f"{X_train.shape=}")
    print(f"{X_test.shape=}")
    print(f"{y_train.shape=}")
    print(f"{y_test.shape=}")

    return X, y, X_train, X_test, y_train, y_test, test_case_id


def get_metrics(y_true: Series, y_pred: Series, y_prob=None) -> AllMetrics:

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    metrics: Metric = {
        "accuracy": accuracy_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob) if y_prob is not None else None,
        "true_positive": tp,
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
    }
    #return metrics
    
    # Get class report to be able to make distinction between nTNBC and TNBC
    classReport = classification_report(y_true, y_pred, output_dict=True, target_names=targetNames, zero_division=0)
    resultCombined = {'metrics': metrics, 'classReport': classReport}
    return resultCombined

def get_cross_validation_metrics(model: Model, X: DataFrame, y: Series, cv: int = 5) -> DataFrame:
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    results: list[Metric] = []

    for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

        # Scale per fold (fit on fold-train only)
        scaler = StandardScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_val_fold_scaled   = scaler.transform(X_val_fold)

        model.fit(X_train_fold_scaled, y_train_fold)
        y_pred_fold = model.predict(X_val_fold_scaled)
        y_prob_fold = model.predict_proba(X_val_fold_scaled)[:, 1]
        
        metrics: Metric = get_metrics(y_val_fold, y_pred_fold, y_prob_fold)
        metrics["fold"] = fold + 1 # ID 0 will be used for the initial testing data
        results.append(metrics)

    df: DataFrame = DataFrame(results)
    df.set_index("fold", inplace=True)
    return df

def get_cross_validation_metrics_smote(model: Model, X: DataFrame, y: Series, cv: int = 5) -> DataFrame:
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    results: list[Metric] = []

    for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

        # Scale per fold (fit on fold-train only)
        scaler = StandardScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_val_fold_scaled   = scaler.transform(X_val_fold)

        # SMOTE per fold (on scaled fold-train only)
        X_train_fold_scaled, y_train_fold = SMOTE(random_state=42).fit_resample(
            X_train_fold_scaled, y_train_fold
        )
        
        # Model
        model.fit(X_train_fold_scaled, y_train_fold)
        y_pred_fold = model.predict(X_val_fold_scaled)
        y_prob_fold = model.predict_proba(X_val_fold_scaled)[:, 1]
        
        metrics: Metric = get_metrics(y_val_fold, y_pred_fold, y_prob_fold)
        metrics["fold"] = fold + 1 # ID 0 will be used for the initial testing data
        results.append(metrics)

    df: DataFrame = DataFrame(results)
    df.set_index("fold", inplace=True)
    return df

def get_cross_validation_metrics_weighted(model: Model, X: DataFrame, y: Series, cv: int = 5) -> DataFrame:
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    results: list[Metric] = []

    # NOTE: 'model' parameter is unused by design. A new fold‑specific
    # weighted model is instantiated per CV fold.   
    for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

        # Scale per fold (fit on fold-train only)
        scaler = StandardScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_val_fold_scaled   = scaler.transform(X_val_fold)

        weights = getDataSetWeights(y_train_fold)
        model_weighted = getWeightedModel(modelName, weights)
        
        # Model
        model_weighted.fit(X_train_fold_scaled, y_train_fold)
        y_pred_fold = model_weighted.predict(X_val_fold_scaled)
        y_prob_fold = model_weighted.predict_proba(X_val_fold_scaled)[:, 1]
        
        metrics: Metric = get_metrics(y_val_fold, y_pred_fold, y_prob_fold)
        metrics["fold"] = fold + 1 # ID 0 will be used for the initial testing data
        results.append(metrics)

    df: DataFrame = DataFrame(results)
    df.set_index("fold", inplace=True)
    return df

def print_evaluated_model_accuracy(model_name:str, y_test: Series, y_pred: Series) -> None:
    print(f"{model_name} - - Accuracy: {accuracy_score(y_test, y_pred):.2f}")    

def print_validated_model_accuracy(model: Model, metrics: DataFrame) -> DataFrame:
    print(f"\nModel validation for {type(model).__name__}:")
    print(metrics)
    #accuracy = metrics['metrics']["accuracy"]
    #print(accuracy.to_list())
    #print(f"\nMean accuracy: {accuracy.mean():.4f}\n")
    return metrics

# Models
def run_model(model: Model, X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: pd.Series, y_test: pd.Series, test_case_ids: pd.Series, is_smote: bool, is_weighted: bool, variant:FeatureVariant):
    # Train the model
    model.fit(X_train, y_train)

    # Model predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]  # For ROC curves etc.

    # Save it in a dataframe, to CSV
    predictions = pd.DataFrame({
        "case_id": test_case_ids,
        "y_test": y_test,
        "y_pred": y_pred,
        "y_prob": y_prob
    })
    predictions.to_csv(f"../Data/output/model_output_{variant}_{GENE_FILE_VARIANT}{'_smote' if is_smote else ''}{'_weighted' if is_weighted else ''}.csv", index=False)

    return y_pred, y_prob

def run_cross_validation(model: Model, X: pd.DataFrame, y: pd.Series, y_test: pd.Series, y_pred: pd.Series, y_prob: pd.Series, is_smote: bool, is_weighted: bool, variant:str) -> pd.DataFrame:
    
    if (is_smote):
        metrics: pd.DataFrame = get_cross_validation_metrics_smote(model, X, y, cv=5)
    elif (is_weighted):
        metrics: pd.DataFrame = get_cross_validation_metrics_weighted(model, X, y, cv=5)
    else:
        metrics: pd.DataFrame = get_cross_validation_metrics(model, X, y, cv=5)
        
    test_metrics = get_metrics(y_test, y_pred, y_prob)
    test_metrics["fold"] = 0 # Initial test metrics (before cross validation)
    test = pd.DataFrame([test_metrics])
    test.set_index("fold", inplace=True)

    print_validated_model_accuracy(model, metrics)

    # Prepend test_metrics to metrics dataframe, export and display
    metrics = pd.concat([test, metrics])

    # parquet to prevent dataloss, json gave dataloss
    metrics.to_parquet(f"../Data/output/model_metrics_{variant}_{GENE_FILE_VARIANT}{'_smote' if is_smote else ''}{'_weighted' if is_weighted else ''}.parquet", engine="pyarrow", index=True)
    #metrics.to_csv(f"../Data/model_metrics_{variant}_{GENE_FILE_VARIANT}{'_smote' if is_smote else ''}{'_weighted' if is_weighted else ''}.csv", index=False)
    return metrics

# Select model
def getModel(value: ModelVariant) -> Model:
    options: dict[str, Model] = {
        'SVM': SVC(random_state=RANDOM_STATE, probability=True),
        'RF': RandomForestClassifier(random_state=RANDOM_STATE),
        'LG': LogisticRegression(random_state=RANDOM_STATE, solver='lbfgs', max_iter=100_000)
    }
    return options.get(value, "Invalid option")

# Select model
def getWeightedModel(value, weights):  
    options = {
        'SVM': SVC(random_state=RANDOM_STATE, class_weight=weights, probability=True),
        'RF': RandomForestClassifier(random_state=RANDOM_STATE, class_weight=weights),
        'LG': LogisticRegression(random_state=RANDOM_STATE, class_weight=weights, solver='lbfgs', max_iter=100_000)
    }
    return options.get(value, "Invalid option")
    
def getDataSetWeights(trainSet: pd.Series):
    from sklearn.utils.class_weight import compute_class_weight
    
    classes = np.unique(trainSet)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=trainSet)
    return dict(zip(classes, weights))


# helper for evaluation charts
def label_bars_right_of_axes(ax, bars, fmt="{:.2f}%", x_axes=1.01, fontsize=9):
    """
    Put labels at a fixed position just outside the right edge of the axes,
    independent of bar width.
    """
    for b in bars:
        y = b.get_y() + b.get_height() / 2
        value = b.get_width()
        ax.text(
            x_axes, y,
            fmt.format(value),
            transform=ax.get_yaxis_transform(),  # x in axes coords, y in data coords
            ha="left", va="center",
            fontsize=fontsize,
            clip_on=False  # allow text outside axes
        )


def label_bars_inside(ax, rects, fmt="{:.2f}%", fontsize=10, color="white"):
    for r in rects:
        w = r.get_width()
        y = r.get_y() + r.get_height() / 2
        ax.text(
            w * 0.5,
            y,
            fmt.format(w),
            va="center",
            ha="left",
            fontsize=fontsize,
            color=color,
            # fontweight="bold"
        )

def lime_aggregate_to_df(aggregate_dict, colDict, model_name, outcome_label):
    """
    Converts an aggregate LIME dictionary into a tidy DataFrame.
    """
    if len(aggregate_dict) == 0:
        return pd.DataFrame()

    df = (
        pd.DataFrame.from_dict(aggregate_dict)
        .rename(columns=colDict)
        .describe()
        .T[['mean']]
        .rename(columns={'mean': 'lime_weight_mean'})
        .reset_index()
        .rename(columns={'index': 'feature'})
    )

    df['model'] = model_name
    df['outcome'] = outcome_label

    return df